SQL Concepts with SAP HANA Cloud using Python hdbcli
This notebook is designed for beginners who want to learn SQL concepts by connecting Python to SAP HANA Cloud using the hdbcli library.

You will learn:

How to connect Python to SAP HANA Cloud
How to create tables
How to insert data
How to read data using SELECT
How to filter, sort, and aggregate data
How to use GROUP BY
How to use joins
How to update and delete records
How to create a view
How to close the database connection safely
Replace the database credentials with your own SAP HANA Cloud details before running the notebook.

1. Install and Import Required Library
hdbcli is the official Python client used to connect Python programs with SAP HANA.

Run the installation cell only once in your environment.

In [1]:
pip install hdbcli

Defaulting to user installation because normal site-packages is not writeable
  Using cached hdbcli-2.29.23-cp310-abi3-win_amd64.whl.metadata (6.3 kB)
Using cached hdbcli-2.29.23-cp310-abi3-win_amd64.whl (3.7 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


2. Connect to SAP HANA Cloud
From your SAP HANA Database Explorer properties screen, you identified:

Host = 13b7c15d-xxxx-xxxx-xxxx-c9c36ab85f56.hna1.prod-eu10.hanacloud.ondemand.com
Port = 443
Database = H00
For hdbcli, normally you need:

address: SAP HANA Cloud host name
port: usually 443
user: database user, for example DBADMIN
password: your database password
encrypt=True: required for SAP HANA Cloud

In [ ]:

HANA_HOST = "13b7c15d-xxxx-xxxx-xxxx-c9c36ab85f56.hna1.prod-eu10.hanacloud.ondemand.com"
HANA_PORT = 443
HANA_USER = "DBADMIN"
HANA_PASSWORD = "enter_your_password_here"

conn = dbapi.connect(
    address = HANA_HOST,
    port = HANA_PORT,
    user = HANA_USER,
    password = HANA_Password,
    encrypt = True
    sslValidateCertificate = False
)

cursor = conn.cursor()
print("Connected successfully to SAP HANA Cloud")

3. Test the Connection

DUMMY is a special one-row table in SAP HANA.
It is commonly used to test SQL queries.

In [ ]:
cursor.execute("SELECT CURRENT_USER, CURRENT_SCHEMA FROM DUMMY")
result = cursor.fetchone()

print("Current User:", result[0])
print("Current Schema:", result[1])

4. Create Tables

We will create three tables:

STUDENTS
Stores student information.

COURSES
Stores course information.

ENROLLMENTS
Connects students with courses.

This is a common relational database design:

STUDENTS  ----<  ENROLLMENTS  >----  COURSES
One student can enroll in many courses.
One course can have many students.

In [ ]:
# Drop existing tables if they already exist
# We drop child table first because it has foreign keys.

tables_to_drop = ["ENROLLMENTS", "STUDENTS", "COURSES"]

for table in tables_to_drop:
    try:
        cursor.execute(f"DROP TABLE {table}")
        print(f"Dropped table: {table}")
    except Exception as e:
        print(f"Table {table} does not exist or cannot be dropped. Skipping.")

conn.commit()

In [ ]:
# Create STUDENTS table

cursor.execute("""
CREATE COLUMN TABLE STUDENTS (
    STUDENT_ID INTEGER PRIMARY KEY,
    STUDENT_NAME NVARCHAR(100) NOT NULL,
    CITY NVARCHAR(50),
    AGE INTEGER
)
""")

# Create COURSES table

cursor.execute("""
CREATE COLUMN TABLE COURSES (
    COURSE_ID INTEGER PRIMARY KEY,
    COURSE_NAME NVARCHAR(100) NOT NULL,
    FEES DECIMAL(10,2)
)
""")

# Create ENROLLMENTS table
# This table uses a composite primary key: STUDENT_ID + COURSE_ID

cursor.execute("""
CREATE COLUMN TABLE ENROLLMENTS (
    STUDENT_ID INTEGER NOT NULL,
    COURSE_ID INTEGER NOT NULL,
    ENROLLMENT_DATE DATE,
    MARKS INTEGER,
    PRIMARY KEY (STUDENT_ID, COURSE_ID),
    FOREIGN KEY (STUDENT_ID) REFERENCES STUDENTS(STUDENT_ID),
    FOREIGN KEY (COURSE_ID) REFERENCES COURSES(COURSE_ID)
)
""")

conn.commit()

print("Tables created successfully")

5. SQL Concept: Primary Key, Foreign Key, and Composite Key

Primary Key
A primary key uniquely identifies each record in a table.

Example:

STUDENT_ID INTEGER PRIMARY KEY
This means two students cannot have the same STUDENT_ID.

Foreign Key
A foreign key creates a relationship between two tables.

Example:

FOREIGN KEY (STUDENT_ID) REFERENCES STUDENTS(STUDENT_ID)
This means an enrollment can only be created for a student that already exists in the STUDENTS table.

Composite Key
A composite key is made from more than one column.

Example:

PRIMARY KEY (STUDENT_ID, COURSE_ID)
This means the same student cannot be enrolled in the same course twice.

6. Insert Data into Tables

INSERT INTO is used to add records into a table.

We will insert:

5 students
4 courses
multiple enrollments

In [ ]:
# Insert data into STUDENTS

students = [
    (1, "Amit Kumar", "Patna", 22),
    (2, "Priya Singh", "Delhi", 21),
    (3, "Rahul Sharma", "Patna", 23),
    (4, "Neha Gupta", "Kolkata", 20),
    (5, "Saurabh Verma", "Bangalore", 24)
]

cursor.executemany(
    "INSERT INTO STUDENTS (STUDENT_ID, STUDENT_NAME, CITY, AGE) VALUES (?, ?, ?, ?)",
    students
)

# Insert data into COURSES

courses = [
    (101, "SQL Basics", 5000.00),
    (102, "Python for Data Analysis", 8000.00),
    (103, "SAP HANA Cloud", 12000.00),
    (104, "Data Visualization", 7000.00)
]

cursor.executemany(
    "INSERT INTO COURSES (COURSE_ID, COURSE_NAME, FEES) VALUES (?, ?, ?)",
    courses
)

# Insert data into ENROLLMENTS

enrollments = [
    (1, 101, "2026-07-01", 85),
    (1, 103, "2026-07-02", 78),
    (2, 101, "2026-07-01", 90),
    (2, 102, "2026-07-03", 88),
    (3, 103, "2026-07-04", 72),
    (4, 104, "2026-07-05", 95),
    (5, 102, "2026-07-06", 80),
    (5, 103, "2026-07-06", 83)
]

cursor.executemany(
    "INSERT INTO ENROLLMENTS (STUDENT_ID, COURSE_ID, ENROLLMENT_DATE, MARKS) VALUES (?, ?, ?, ?)",
    enrollments
)

conn.commit()

print("Sample data inserted successfully")

7. Helper Function to Display Query Results

This helper function runs a SQL query and prints the result in a readable format.

In [ ]:
def run_query(sql):
    cursor.execute(sql)
    rows = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]

    print(" | ".join(columns))
    print("-" * 80)

    for row in rows:
        print(" | ".join(str(value) for value in row))

8. SELECT: Read Data from a Table

SELECT is used to read data from a database table.

In [ ]:
run_query("SELECT * FROM STUDENTS")

9. Select Specific Columns

Instead of selecting all columns using *, you can select only the columns you need.

In [ ]:
run_query("""
SELECT STUDENT_ID, STUDENT_NAME, CITY
FROM STUDENTS
""")


10. WHERE: Filter Records

WHERE is used to apply conditions.

Example: show only students from Patna.

In [ ]:
run_query("""
SELECT *
FROM STUDENTS
WHERE CITY = 'Patna'
""")

11. ORDER BY: Sort Records

ORDER BY is used to sort results.

Example: show students sorted by age.

In [ ]:
run_query("""
SELECT *
FROM STUDENTS
ORDER BY AGE DESC
""")

12. Aggregate Functions

Aggregate functions calculate summary values.

Common aggregate functions:

Function	Meaning
COUNT()	Counts records
SUM()	Adds values
AVG()	Calculates average
MIN()	Finds minimum
MAX()	Finds maximum


In [ ]:
run_query("""
SELECT 
    COUNT(*) AS TOTAL_STUDENTS,
    AVG(AGE) AS AVERAGE_AGE,
    MIN(AGE) AS MIN_AGE,
    MAX(AGE) AS MAX_AGE
FROM STUDENTS
""")

13. GROUP BY: Group Records

GROUP BY groups records based on one or more columns.

Example: count students city-wise.

In [ ]:
run_query("""
SELECT 
    CITY,
    COUNT(*) AS TOTAL_STUDENTS
FROM STUDENTS
GROUP BY CITY
ORDER BY TOTAL_STUDENTS DESC
""")

14. HAVING: Filter Grouped Results

WHERE filters rows before grouping.
HAVING filters groups after grouping.

Example: show cities where more than one student is present.

In [ ]:
run_query("""
SELECT 
    CITY,
    COUNT(*) AS TOTAL_STUDENTS
FROM STUDENTS
GROUP BY CITY
HAVING COUNT(*) > 1
""")

15. INNER JOIN

INNER JOIN returns only matching records from both tables.

Example: show student names with their enrolled courses.

In [ ]:
run_query("""
SELECT 
    S.STUDENT_ID,
    S.STUDENT_NAME,
    C.COURSE_NAME,
    E.ENROLLMENT_DATE,
    E.MARKS
FROM ENROLLMENTS E
INNER JOIN STUDENTS S
    ON E.STUDENT_ID = S.STUDENT_ID
INNER JOIN COURSES C
    ON E.COURSE_ID = C.COURSE_ID
ORDER BY S.STUDENT_ID
""")

16. LEFT JOIN

LEFT JOIN returns all records from the left table and matching records from the right table.

Example: show all students and their courses if available.

In [ ]:
run_query("""
SELECT 
    S.STUDENT_ID,
    S.STUDENT_NAME,
    C.COURSE_NAME
FROM STUDENTS S
LEFT JOIN ENROLLMENTS E
    ON S.STUDENT_ID = E.STUDENT_ID
LEFT JOIN COURSES C
    ON E.COURSE_ID = C.COURSE_ID
ORDER BY S.STUDENT_ID
""")

17. UPDATE: Modify Existing Data

UPDATE is used to change existing records.

Example: update the marks of one student in one course.

In [ ]:
cursor.execute("""
UPDATE ENROLLMENTS
SET MARKS = 92
WHERE STUDENT_ID = 1
  AND COURSE_ID = 101
""")

conn.commit()

print("Record updated successfully")

run_query("""
SELECT *
FROM ENROLLMENTS
WHERE STUDENT_ID = 1
  AND COURSE_ID = 101
""")


18. DELETE: Remove Records

DELETE removes records from a table.

For safety, always use a WHERE condition.

In [ ]:
# Example: delete one enrollment record

cursor.execute("""
DELETE FROM ENROLLMENTS
WHERE STUDENT_ID = 4
  AND COURSE_ID = 104
""")

conn.commit()

print("Record deleted successfully")

run_query("SELECT * FROM ENROLLMENTS ORDER BY STUDENT_ID, COURSE_ID")

19. Create a View

A view is a saved SQL query.

It does not store data separately like a table.
It helps simplify complex queries.

Example: create a view that shows student-course enrollment details.

In [ ]:
# Drop view if it already exists

try:
    cursor.execute("DROP VIEW STUDENT_COURSE_VIEW")
    conn.commit()
    print("Old view dropped.")
except Exception:
    print("View does not exist. Creating new view.")

cursor.execute("""
CREATE VIEW STUDENT_COURSE_VIEW AS
SELECT 
    S.STUDENT_ID,
    S.STUDENT_NAME,
    S.CITY,
    C.COURSE_ID,
    C.COURSE_NAME,
    C.FEES,
    E.ENROLLMENT_DATE,
    E.MARKS
FROM ENROLLMENTS E
INNER JOIN STUDENTS S
    ON E.STUDENT_ID = S.STUDENT_ID
INNER JOIN COURSES C
    ON E.COURSE_ID = C.COURSE_ID
""")

conn.commit()

print("View created successfully")

In [ ]:
run_query("""
SELECT *
FROM STUDENT_COURSE_VIEW
ORDER BY STUDENT_ID, COURSE_ID
""")

20. Practice Questions

Try writing SQL queries for the following:

Show all students whose age is greater than 21.
Show total enrollment count for each course.
Show average marks for each course.
Show students who enrolled in SAP HANA Cloud.
Show the highest marks scored by each student.
Create a view for course-wise performance.

21. Close the Connection

Always close the cursor and connection after your work is done.

In [ ]:
cursor.close()
conn.close()

print("Connection closed successfully")